In [17]:
import pandas as pd
import numpy as np
import pickle

In [18]:
pickle_file ="results/optimal_portfolios/optimal_portfolios_all_te_0623.pkl"

with open(pickle_file, "rb") as f:
        sector_weights = pickle.load(f)


In [19]:
sector_weights['Industrials'].keys()

dict_keys(['cov_type', 'diagnostics', 'weights_by_te', 'tracking_errors', 'carbon_reductions', 'w_bench', 'stock_labels'])

In [20]:
s = sector_weights['Industrials']['tracking_errors']
closest_idx = min(range(len(s)), key=lambda i: abs(s[i] - 200))


In [21]:
sector_weights['Industrials']['carbon_reductions'][closest_idx]

92.16792334413321

In [22]:
import pandas as pd

labels = sector_weights['Industrials']['stock_labels']
w_bench = sector_weights['Industrials']['w_bench']
w_opt   = sector_weights['Industrials']['weights_by_te'][closest_idx]

weight_differences = pd.DataFrame({
    'stock': labels,
    'w_bench': w_bench,
    'w_opt': w_opt,
})

# decrease = benchmark - optimized
weight_differences['diff'] = weight_differences['w_opt'] - weight_differences['w_bench']

weight_differences['w_opt'] = weight_differences['w_opt'].round(6)
weight_differences['w_bench'] = weight_differences['w_bench'].round(6)
weight_differences['diff'] = weight_differences['diff'].round(6)

# sort: highest decrease first
weight_differences = weight_differences.sort_values('diff')

data_file = f"data/datasets/dataset_comp_0623.xlsx"
df = pd.read_excel(data_file)

# order of stocks based on weight_differences sorting
order = weight_differences['stock'].tolist()

df_ind = df.loc[df['GICS Sector'] == 'Industrials'].copy()

df_ind['SYMBOL'] = pd.Categorical(
    df_ind['SYMBOL'],
    categories=order,
    ordered=True
)

df_ind = df_ind.sort_values('SYMBOL')


In [23]:
weight_differences

,stock,w_bench,w_opt,diff
2,ADP,0.044463,0.000000,-0.044463
7,URI,0.037447,0.000000,-0.037447
10,CSX,0.024674,0.000000,-0.024674
0,ALK,0.048422,0.025445,-0.022977
12,CTAS,0.021578,0.000000,-0.021578
...,...,...,...,...
16,RSG,0.018212,0.051955,0.033743
11,CAT,0.022460,0.057672,0.035212
32,NSC,0.010519,0.048033,0.037514
63,OTIS,0.003836,0.043603,0.039767


In [24]:
df_ind = pd.merge(df_ind, weight_differences, how = 'left', left_on = 'SYMBOL', right_on = 'stock')

In [25]:
df_ind['CI_contrib_bench'] = df_ind['w_bench'] * df_ind['Carbon Intensity']
df_ind['CI_contrib_opt']   = df_ind['w_opt']   * df_ind['Carbon Intensity']

df_ind['CI_contrib_diff'] = df_ind['CI_contrib_opt'] - df_ind['CI_contrib_bench']


In [26]:
df_sorted = df_ind.sort_values('CI_contrib_bench', ascending=False).copy()

total = df_sorted['CI_contrib_bench'].sum()

df_sorted['cum_pct'] = df_sorted['CI_contrib_bench'].cumsum() / total

top_90 = df_sorted[df_sorted['cum_pct'] <= 0.90]

top_90[['CI_contrib_bench', 'w_bench', 'w_opt', 'cum_pct']]


,CI_contrib_bench,w_bench,w_opt,cum_pct
6,0.655778,0.018977,0.000000,0.391692
43,0.098801,0.003107,0.000000,0.450706
55,0.081548,0.003929,0.012087,0.499414
17,0.077298,0.010719,0.000000,0.545584
41,0.062063,0.003735,0.000000,0.582653
74,0.049454,0.013818,0.055088,0.612192
3,0.047557,0.048422,0.025445,0.640597
32,0.046548,0.005402,0.000000,0.668400
25,0.041014,0.040031,0.032690,0.692898
28,0.036471,0.006668,0.000000,0.714681


In [27]:
len(top_90)

22

In [28]:
top_90['w_bench'].sum()

0.348885

In [29]:
top_90.filter(regex = 'Imputed|SYMBOL|Carbon Intensity|Filled')

,SYMBOL,Filled Scope 1 Count,Filled Scope 2 Count,Filled Scope 3 Count,Carbon Intensity,Scope 1 Imputed,Scope 2 Imputed,Scope 3 Imputed
6,CMI,0,0,0,34.556488,0,0,0
43,IR,0,0,0,31.799511,0,0,0
55,CARR,0,0,0,20.755523,0,0,0
17,NDSN,0,0,0,7.211319,1,1,1
41,TT,0,0,0,16.616504,0,0,0
74,GWW,0,0,0,3.578954,0,0,0
3,ALK,0,0,0,0.982142,0,0,0
32,XYL,0,0,0,8.616741,0,0,0
25,CHRW,0,0,0,1.024559,0,0,0
28,WAB,0,0,0,5.469563,0,0,0


So I dove deeper and looked at some crucial info for the Industrials sectors. What I found out is that 22 stocks account for 90% of the benchmakr carbon intensity contribution and they account for the 34% of the weight in the benchmark portfolio. Therefore do you think it is plausible that with only a 2% TE, we obtain a 92% reduction in weighted carbon intensity? Given that the annualized volatility of this sector is: 17.25% (so actually lower than the avg. sector volatility that is 20.56%). 

In [30]:
te_results_0623_next_3m = pd.read_excel("results/robustness/te_results_0622_next_3m.xlsx")

In [31]:
import pandas as pd
import glob

# Get all te_results files
files = glob.glob('results/robustness/te_results_*_next_3m.xlsx')

# Read and concatenate all files
all_dfs = []
for f in files:
    df = pd.read_excel(f)
    period = f.split('te_results_')[1].split('_next')[0]  # Extract period code
    df['period'] = period
    all_dfs.append(df)

combined = pd.concat(all_dfs, ignore_index=True)

# Calculate average annualized_TE by sector across all periods
avg_te = combined.groupby('sector')['annualized_TE'].mean()
print(avg_te)

# Or if you want the overall average across all sectors and periods:
overall_avg = combined['annualized_TE'].mean()
print(f"\nOverall average annualized_TE: {overall_avg:.6f}")

sector
Communication Services    0.170257
Consumer Discretionary    0.093222
Consumer Staples          0.053245
Energy                    0.081650
Financials                0.054925
Health Care               0.066819
Industrials               0.043014
Information Technology    0.110760
Materials                 0.052364
Real Estate               0.062366
Utilities                 0.051655
Name: annualized_TE, dtype: float64

Overall average annualized_TE: 0.076389


In [625]:
combined.loc[combined['sector']!='Communication Services', 'annualized_TE'].mean()

0.03342466403658712

In [455]:
avg_te.sort_values()

sector
Industrials               0.019027
Utilities                 0.024729
Materials                 0.025656
Financials                0.027145
Consumer Staples          0.029750
Real Estate               0.032670
Health Care               0.033229
Energy                    0.037693
Consumer Discretionary    0.046575
Information Technology    0.055277
Communication Services    0.084954
Name: annualized_TE, dtype: float64

In [56]:
te_results_0623_next_3m.sort_values(by = 'annualized_TE', ascending=True)

,sector,annualized_TE
6,Industrials,0.042844
10,Utilities,0.044147
2,Consumer Staples,0.049221
8,Materials,0.050354
4,Financials,0.050535
9,Real Estate,0.064703
5,Health Care,0.080177
1,Consumer Discretionary,0.094992
3,Energy,0.099109
7,Information Technology,0.112637
